# Notebook 05: Product Utilization Impact Analysis

This notebook analyzes how product usage, product count, and product depth influence customer retention and churn behavior.

The goal is to understand whether customers with more banking products are more loyal, whether single-product customers are more vulnerable, and whether product depth creates stronger retention.

### Import Required Libraries

In [22]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


### Define Project Paths

In [23]:
# Define project paths
BASE_DIR = Path.cwd().parent

FEATURES_DATA_PATH = BASE_DIR / "data" / "processed" / "european_bank_features.csv"

PRODUCT_OUTPUT_DIR = BASE_DIR / "data" / "processed" / "product_utilization_analysis"

print("Base Directory:", BASE_DIR)
print("Feature-Engineered Data Path:", FEATURES_DATA_PATH)
print("Product Output Directory:", PRODUCT_OUTPUT_DIR)

Base Directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy
Feature-Engineered Data Path: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\european_bank_features.csv
Product Output Directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\product_utilization_analysis


### Load Feature-Engineered Dataset

In [24]:
# Load feature-engineered dataset
df = pd.read_csv(FEATURES_DATA_PATH)

# Preview first rows
df.head()

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Churn_Status,Engagement_Status,Credit_Card_Status,Product_Depth_Segment,Balance_Segment,Salary_Segment,Age_Segment,Tenure_Segment,High_Value_Customer_Flag,High_Value_Customer_Status,High_Value_Disengaged_Flag,High_Value_Disengaged_Status,At_Risk_Premium_Flag,At_Risk_Premium_Status,Salary_Balance_Mismatch_Flag,Salary_Balance_Mismatch_Status,Customer_Risk_Segment,Relationship_Strength_Index,Relationship_Strength_Tier
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,"101,348.88",1,Churned,Active Member,Has Credit Card,Single Product,Zero Balance,High Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,55,Moderate Relationship
1,2025,15647311,Hill,608,Spain,Female,41,1,"83,807.86",1,0,1,"112,542.58",0,Retained,Active Member,No Credit Card,Single Product,Medium Balance,High Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,55,Moderate Relationship
2,2025,15619304,Onio,502,France,Female,42,8,"159,660.80",3,1,0,"113,931.57",1,Churned,Inactive Member,Has Credit Card,High Product Load,Premium Balance,High Salary,Mid-Age Customer,Established Relationship,1,High Value Customer,1,High-Value Disengaged,1,At-Risk Premium Customer,0,No Salary-Balance Mismatch,At-Risk Premium,50,Weak Relationship
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,"93,826.63",0,Retained,Inactive Member,No Credit Card,Optimal Product Depth,Zero Balance,Medium Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,35,Weak Relationship
4,2025,15737888,Mitchell,850,Spain,Female,43,2,"125,510.82",1,1,1,"79,084.10",0,Retained,Active Member,Has Credit Card,Single Product,High Balance,Medium Salary,Mid-Age Customer,New Customer,0,Standard Value Customer,0,Not High-Value Disengaged,0,Not At-Risk Premium,0,No Salary-Balance Mismatch,Standard Monitoring,75,Moderate Relationship


### Dataset Shape Check

In [25]:
# Check dataset shape
print(f"Total Rows: {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]:,}")

Total Rows: 10,000
Total Columns: 33


### Product Count Distribution

This step checks how many customers are using 1, 2, 3, or 4 banking products.

In [26]:
# Analyze product count distribution
product_count_distribution = (
    df["NumOfProducts"]
    .value_counts()
    .sort_index()
    .reset_index()
)

product_count_distribution.columns = ["Number of Products", "Customer Count"]

product_count_distribution["Customer %"] = (
    product_count_distribution["Customer Count"] / len(df) * 100
).round(2)

product_count_distribution

,Number of Products,Customer Count,Customer %
0,1,5084,50.84
1,2,4590,45.90
2,3,266,2.66
3,4,60,0.60


### Product Count vs Churn Analysis

This step measures churn rate for each product count group.

In [27]:
# Analyze churn rate by number of products
product_count_churn = (
    df
    .groupby("NumOfProducts")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Churn_Rate=("Exited", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

product_count_churn["Churn_Rate"] = (
    product_count_churn["Churn_Rate"] * 100
).round(2)

product_count_churn["Retention_Rate"] = (
    product_count_churn["Retained_Customers"] / product_count_churn["Customer_Count"] * 100
).round(2)

product_count_churn["Avg_Relationship_Strength"] = (
    product_count_churn["Avg_Relationship_Strength"]
).round(2)

product_count_churn = product_count_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

product_count_churn

,NumOfProducts,Customer_Count,Churned_Customers,Retained_Customers,Churn_Rate,Avg_Relationship_Strength,Retention_Rate
3,4,60,60,0,100.00,52.67,0.00
2,3,266,220,46,82.71,47.74,17.29
0,1,5084,1409,3675,27.71,58.30,72.29
1,2,4590,348,4242,7.58,72.37,92.42


## Single-Product vs Multi-Product Segmentation

This step separates customers into single-product and multi-product groups to compare retention behavior.

In [28]:
# Create single-product vs multi-product segment
df["Product_Usage_Group"] = np.where(
    df["NumOfProducts"] == 1,
    "Single Product Customer",
    "Multi Product Customer"
)

product_usage_distribution = (
    df["Product_Usage_Group"]
    .value_counts()
    .reset_index()
)

product_usage_distribution.columns = ["Product Usage Group", "Customer Count"]

product_usage_distribution["Customer %"] = (
    product_usage_distribution["Customer Count"] / len(df) * 100
).round(2)

product_usage_distribution

,Product Usage Group,Customer Count,Customer %
0,Single Product Customer,5084,50.84
1,Multi Product Customer,4916,49.16


### Single-Product vs Multi-Product Churn Analysis

This step compares churn behavior between single-product and multi-product customers.

In [29]:
# Analyze churn rate by product usage group
product_usage_churn = (
    df
    .groupby("Product_Usage_Group")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Churn_Rate=("Exited", "mean"),
        Avg_Num_Products=("NumOfProducts", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

product_usage_churn["Churn_Rate"] = (
    product_usage_churn["Churn_Rate"] * 100
).round(2)

product_usage_churn["Retention_Rate"] = (
    product_usage_churn["Retained_Customers"] / product_usage_churn["Customer_Count"] * 100
).round(2)

product_usage_churn["Avg_Num_Products"] = product_usage_churn["Avg_Num_Products"].round(2)
product_usage_churn["Avg_Relationship_Strength"] = product_usage_churn["Avg_Relationship_Strength"].round(2)

product_usage_churn = product_usage_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

product_usage_churn

,Product_Usage_Group,Customer_Count,Churned_Customers,Retained_Customers,Churn_Rate,Avg_Num_Products,Avg_Relationship_Strength,Retention_Rate
1,Single Product Customer,5084,1409,3675,27.71,1.00,58.30,72.29
0,Multi Product Customer,4916,628,4288,12.77,2.08,70.80,87.23


### Product Depth Segment Distribution

This step uses the Product_Depth_Segment created in Notebook 03 to understand customer distribution across product depth groups.

In [30]:
# Analyze product depth segment distribution
product_depth_distribution = (
    df["Product_Depth_Segment"]
    .value_counts()
    .reset_index()
)

product_depth_distribution.columns = ["Product Depth Segment", "Customer Count"]

product_depth_distribution["Customer %"] = (
    product_depth_distribution["Customer Count"] / len(df) * 100
).round(2)

product_depth_distribution

,Product Depth Segment,Customer Count,Customer %
0,Single Product,5084,50.84
1,Optimal Product Depth,4590,45.90
2,High Product Load,326,3.26


## Product Depth Segment vs Churn Analysis

This step compares churn and retention behavior across product depth segments created during feature engineering.

The purpose is to identify whether customers with single product, optimal product depth, or high product load show different churn patterns.

In [31]:
# Analyze churn behavior by product depth segment
product_depth_churn = (
    df
    .groupby("Product_Depth_Segment")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Num_Products=("NumOfProducts", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

product_depth_churn["Churn_Rate"] = (
    product_depth_churn["Churned_Customers"] / product_depth_churn["Customer_Count"] * 100
).round(2)

product_depth_churn["Retention_Rate"] = (
    product_depth_churn["Retained_Customers"] / product_depth_churn["Customer_Count"] * 100
).round(2)

product_depth_churn["Avg_Num_Products"] = product_depth_churn["Avg_Num_Products"].round(2)
product_depth_churn["Avg_Relationship_Strength"] = product_depth_churn["Avg_Relationship_Strength"].round(2)

product_depth_churn = product_depth_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

product_depth_churn

,Product_Depth_Segment,Customer_Count,Churned_Customers,Retained_Customers,Avg_Num_Products,Avg_Relationship_Strength,Churn_Rate,Retention_Rate
0,High Product Load,326,280,46,3.18,48.65,85.89,14.11
2,Single Product,5084,1409,3675,1.00,58.30,27.71,72.29
1,Optimal Product Depth,4590,348,4242,2.00,72.37,7.58,92.42


## Product Depth Retention Gap Check

This step calculates the retention gap between optimal product depth customers and other product depth groups.

This helps validate whether having the right number of products improves customer retention.

In [32]:
# Extract retention rates for product depth comparison
optimal_depth_retention = product_depth_churn.loc[
    product_depth_churn["Product_Depth_Segment"] == "Optimal Product Depth",
    "Retention_Rate"
].iloc[0]

single_product_retention = product_depth_churn.loc[
    product_depth_churn["Product_Depth_Segment"] == "Single Product",
    "Retention_Rate"
].iloc[0]

high_product_load_retention = product_depth_churn.loc[
    product_depth_churn["Product_Depth_Segment"] == "High Product Load",
    "Retention_Rate"
].iloc[0]

optimal_vs_single_gap = optimal_depth_retention - single_product_retention
optimal_vs_high_load_gap = optimal_depth_retention - high_product_load_retention

print(f"Optimal Product Depth Retention Rate: {optimal_depth_retention:.2f}%")
print(f"Single Product Retention Rate: {single_product_retention:.2f}%")
print(f"High Product Load Retention Rate: {high_product_load_retention:.2f}%")
print(f"Retention Gap: Optimal vs Single Product: {optimal_vs_single_gap:.2f} percentage points")
print(f"Retention Gap: Optimal vs High Product Load: {optimal_vs_high_load_gap:.2f} percentage points")

Optimal Product Depth Retention Rate: 92.42%
Single Product Retention Rate: 72.29%
High Product Load Retention Rate: 14.11%
Retention Gap: Optimal vs Single Product: 20.13 percentage points
Retention Gap: Optimal vs High Product Load: 78.31 percentage points


## Product Usage Group vs Churn Analysis

This step compares single-product and multi-product customers to understand whether broader product usage improves retention behavior.

In [33]:
# Analyze churn behavior by product usage group
product_usage_churn = (
    df
    .groupby("Product_Usage_Group")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Num_Products=("NumOfProducts", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

product_usage_churn["Churn_Rate"] = (
    product_usage_churn["Churned_Customers"] / product_usage_churn["Customer_Count"] * 100
).round(2)

product_usage_churn["Retention_Rate"] = (
    product_usage_churn["Retained_Customers"] / product_usage_churn["Customer_Count"] * 100
).round(2)

product_usage_churn["Avg_Num_Products"] = product_usage_churn["Avg_Num_Products"].round(2)
product_usage_churn["Avg_Relationship_Strength"] = product_usage_churn["Avg_Relationship_Strength"].round(2)

product_usage_churn = product_usage_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

product_usage_churn

,Product_Usage_Group,Customer_Count,Churned_Customers,Retained_Customers,Avg_Num_Products,Avg_Relationship_Strength,Churn_Rate,Retention_Rate
1,Single Product Customer,5084,1409,3675,1.00,58.30,27.71,72.29
0,Multi Product Customer,4916,628,4288,2.08,70.80,12.77,87.23


## Single vs Multi Product Retention Comparison

This step measures how much stronger multi-product customer retention is compared with single-product customer retention.

In [34]:
# Compare retention between single-product and multi-product customers
single_product_rate = product_usage_churn.loc[
    product_usage_churn["Product_Usage_Group"] == "Single Product Customer",
    "Retention_Rate"
].iloc[0]

multi_product_rate = product_usage_churn.loc[
    product_usage_churn["Product_Usage_Group"] == "Multi Product Customer",
    "Retention_Rate"
].iloc[0]

single_product_churn_rate = product_usage_churn.loc[
    product_usage_churn["Product_Usage_Group"] == "Single Product Customer",
    "Churn_Rate"
].iloc[0]

multi_product_churn_rate = product_usage_churn.loc[
    product_usage_churn["Product_Usage_Group"] == "Multi Product Customer",
    "Churn_Rate"
].iloc[0]

multi_product_retention_lift = multi_product_rate - single_product_rate
single_product_churn_multiplier = single_product_churn_rate / multi_product_churn_rate

print(f"Single Product Retention Rate: {single_product_rate:.2f}%")
print(f"Multi Product Retention Rate: {multi_product_rate:.2f}%")
print(f"Multi Product Retention Lift: {multi_product_retention_lift:.2f} percentage points")
print(f"Single Product Churn Risk Multiplier: {single_product_churn_multiplier:.2f}x")

Single Product Retention Rate: 72.29%
Multi Product Retention Rate: 87.23%
Multi Product Retention Lift: 14.94 percentage points
Single Product Churn Risk Multiplier: 2.17x


## Product Depth Churn Contribution Analysis

This step calculates how much each product depth segment contributes to total customer churn.

This is useful for dashboard prioritization because a segment with high churn rate may not always contribute the largest share of total churn.

In [35]:
# Calculate churn contribution by product depth segment
total_churned_customers = df["Exited"].sum()

product_depth_contribution = product_depth_churn.copy()

product_depth_contribution["Churn_Contribution_%"] = (
    product_depth_contribution["Churned_Customers"] / total_churned_customers * 100
).round(2)

product_depth_contribution["Customer_Share_%"] = (
    product_depth_contribution["Customer_Count"] / len(df) * 100
).round(2)

product_depth_contribution = product_depth_contribution[
    [
        "Product_Depth_Segment",
        "Customer_Count",
        "Customer_Share_%",
        "Churned_Customers",
        "Churn_Contribution_%",
        "Churn_Rate",
        "Retention_Rate"
    ]
].sort_values(
    by="Churn_Contribution_%",
    ascending=False
)

product_depth_contribution

,Product_Depth_Segment,Customer_Count,Customer_Share_%,Churned_Customers,Churn_Contribution_%,Churn_Rate,Retention_Rate
2,Single Product,5084,50.84,1409,69.17,27.71,72.29
1,Optimal Product Depth,4590,45.90,348,17.08,7.58,92.42
0,High Product Load,326,3.26,280,13.75,85.89,14.11


## Product Utilization Risk Segment

This step creates a dashboard-ready risk segment based on product usage behavior and churn risk.

The goal is to classify customers into practical product utilization groups that can support retention strategy.

In [36]:
# Create product utilization risk segment
def assign_product_utilization_risk(row):
    if row["Exited"] == 1:
        return "Already Churned"
    elif row["NumOfProducts"] == 4:
        return "Critical Product Load Risk"
    elif row["NumOfProducts"] == 3:
        return "High Product Load Risk"
    elif row["NumOfProducts"] == 1 and row["IsActiveMember"] == 0:
        return "Inactive Single-Product Risk"
    elif row["NumOfProducts"] == 2 and row["IsActiveMember"] == 1:
        return "Optimal Active Product User"
    else:
        return "Standard Product Monitoring"


df["Product_Utilization_Risk_Segment"] = df.apply(
    assign_product_utilization_risk,
    axis=1
)

product_utilization_risk_distribution = (
    df["Product_Utilization_Risk_Segment"]
    .value_counts()
    .reset_index()
)

product_utilization_risk_distribution.columns = [
    "Product Utilization Risk Segment",
    "Customer Count"
]

product_utilization_risk_distribution["Customer %"] = (
    product_utilization_risk_distribution["Customer Count"] / len(df) * 100
).round(2)

product_utilization_risk_distribution

,Product Utilization Risk Segment,Customer Count,Customer %
0,Standard Product Monitoring,4010,40.10
1,Optimal Active Product User,2310,23.10
2,Already Churned,2037,20.37
3,Inactive Single-Product Risk,1597,15.97
4,High Product Load Risk,46,0.46


## Product Utilization Risk vs Churn Analysis

This step validates product utilization risk segments against churn behavior.

It helps identify which product usage profiles should be prioritized in the retention dashboard.

In [37]:
# Analyze churn by product utilization risk segment
product_utilization_risk_churn = (
    df
    .groupby("Product_Utilization_Risk_Segment")
    .agg(
        Customer_Count=("CustomerId", "count"),
        Churned_Customers=("Exited", "sum"),
        Retained_Customers=("Exited", lambda x: (x == 0).sum()),
        Avg_Num_Products=("NumOfProducts", "mean"),
        Avg_Relationship_Strength=("Relationship_Strength_Index", "mean")
    )
    .reset_index()
)

product_utilization_risk_churn["Churn_Rate"] = (
    product_utilization_risk_churn["Churned_Customers"] / product_utilization_risk_churn["Customer_Count"] * 100
).round(2)

product_utilization_risk_churn["Retention_Rate"] = (
    product_utilization_risk_churn["Retained_Customers"] / product_utilization_risk_churn["Customer_Count"] * 100
).round(2)

product_utilization_risk_churn["Avg_Num_Products"] = (
    product_utilization_risk_churn["Avg_Num_Products"].round(2)
)

product_utilization_risk_churn["Avg_Relationship_Strength"] = (
    product_utilization_risk_churn["Avg_Relationship_Strength"].round(2)
)

product_utilization_risk_churn = product_utilization_risk_churn.sort_values(
    by="Churn_Rate",
    ascending=False
)

product_utilization_risk_churn

,Product_Utilization_Risk_Segment,Customer_Count,Churned_Customers,Retained_Customers,Avg_Num_Products,Avg_Relationship_Strength,Churn_Rate,Retention_Rate
0,Already Churned,2037,2037,0,1.48,55.68,100.00,0.00
1,High Product Load Risk,46,0,46,3.00,45.98,0.00,100.00
2,Inactive Single-Product Risk,1597,0,1597,1.00,43.81,0.00,100.00
3,Optimal Active Product User,2310,0,2310,2.00,85.99,0.00,100.00
4,Standard Product Monitoring,4010,0,4010,1.48,64.91,0.00,100.00


## Product Utilization KPI Summary

This step creates a compact KPI summary for dashboard development.

These metrics will help communicate product usage risk and retention opportunity clearly in the final Streamlit dashboard.

In [38]:
# Create product utilization KPI summary
product_utilization_kpi_summary = pd.DataFrame({
    "KPI": [
        "Single Product Customers",
        "Multi Product Customers",
        "Single Product Churn Rate",
        "Multi Product Churn Rate",
        "Multi Product Retention Lift",
        "Single Product Churn Risk Multiplier",
        "Optimal Product Depth Retention Rate",
        "High Product Load Churn Rate"
    ],
    "Value": [
        f"{product_usage_churn.loc[product_usage_churn['Product_Usage_Group'] == 'Single Product Customer', 'Customer_Count'].iloc[0]:,}",
        f"{product_usage_churn.loc[product_usage_churn['Product_Usage_Group'] == 'Multi Product Customer', 'Customer_Count'].iloc[0]:,}",
        f"{single_product_churn_rate:.2f}%",
        f"{multi_product_churn_rate:.2f}%",
        f"{multi_product_retention_lift:.2f} percentage points",
        f"{single_product_churn_multiplier:.2f}x",
        f"{optimal_depth_retention:.2f}%",
        f"{high_product_load_retention:.2f}% retention / {100 - high_product_load_retention:.2f}% churn"
    ],
    "Business Meaning": [
        "Customers using only one banking product",
        "Customers using more than one banking product",
        "Churn rate among single-product customers",
        "Churn rate among multi-product customers",
        "Retention advantage of multi-product customers over single-product customers",
        "How much higher single-product churn risk is compared with multi-product customers",
        "Retention rate for customers with optimal product depth",
        "Risk level among customers with high product load"
    ]
})

product_utilization_kpi_summary

,KPI,Value,Business Meaning
0,Single Product Customers,"5,084",Customers using only one banking product
1,Multi Product Customers,"4,916",Customers using more than one banking product
2,Single Product Churn Rate,27.71%,Churn rate among single-product customers
3,Multi Product Churn Rate,12.77%,Churn rate among multi-product customers
4,Multi Product Retention Lift,14.94 percentage points,Retention advantage of multi-product customers...
5,Single Product Churn Risk Multiplier,2.17x,How much higher single-product churn risk is c...
6,Optimal Product Depth Retention Rate,92.42%,Retention rate for customers with optimal prod...
7,High Product Load Churn Rate,14.11% retention / 85.89% churn,Risk level among customers with high product load


## Product Utilization Insight Table

This step creates dashboard-ready insight statements based on product utilization behavior.

These insights will support the product utilization module in the final Streamlit dashboard.

In [39]:
# Create product utilization insight table
product_utilization_insight_table = pd.DataFrame({
    "Insight Area": [
        "Single Product Risk",
        "Multi Product Retention Strength",
        "Optimal Product Depth",
        "High Product Load Risk",
        "Product Depth Contribution",
        "Dashboard Priority"
    ],
    "Key Finding": [
        "Single-product customers show weaker retention compared with multi-product customers.",
        "Multi-product customers show stronger retention and lower churn risk.",
        "Customers with optimal product depth show the strongest retention profile.",
        "Customers with high product load show elevated churn risk and require monitoring.",
        "Product depth helps explain both churn rate and churn contribution.",
        "Product utilization should be included as a core dashboard segmentation lens."
    ],
    "Metric Evidence": [
        f"Single product churn: {single_product_churn_rate:.2f}%",
        f"Multi product retention: {multi_product_rate:.2f}%",
        f"Optimal product depth retention: {optimal_depth_retention:.2f}%",
        f"High product load retention: {high_product_load_retention:.2f}%",
        "Churn contribution calculated by product depth segment",
        "Product utilization risk segment created"
    ],
    "Dashboard Use": [
        "Single-product customer risk card",
        "Multi-product retention KPI",
        "Best retention profile insight",
        "High product load warning chart",
        "Product depth contribution chart",
        "Product utilization risk table"
    ]
})

product_utilization_insight_table

,Insight Area,Key Finding,Metric Evidence,Dashboard Use
0,Single Product Risk,Single-product customers show weaker retention...,Single product churn: 27.71%,Single-product customer risk card
1,Multi Product Retention Strength,Multi-product customers show stronger retentio...,Multi product retention: 87.23%,Multi-product retention KPI
2,Optimal Product Depth,Customers with optimal product depth show the ...,Optimal product depth retention: 92.42%,Best retention profile insight
3,High Product Load Risk,Customers with high product load show elevated...,High product load retention: 14.11%,High product load warning chart
4,Product Depth Contribution,Product depth helps explain both churn rate an...,Churn contribution calculated by product depth...,Product depth contribution chart
5,Dashboard Priority,Product utilization should be included as a co...,Product utilization risk segment created,Product utilization risk table


## Export Product Utilization Analysis Outputs

This step exports product utilization analysis outputs for dashboard development and future reporting.

In [42]:
# Create product utilization output directory if it does not exist
PRODUCT_OUTPUT_DIR = FEATURES_DATA_PATH.parent / "product_utilization_analysis"
PRODUCT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Export product utilization analysis outputs
product_count_distribution.to_csv(
    PRODUCT_OUTPUT_DIR / "product_count_distribution.csv",
    index=False
)

product_count_churn.to_csv(
    PRODUCT_OUTPUT_DIR / "product_count_churn.csv",
    index=False
)

product_usage_distribution.to_csv(
    PRODUCT_OUTPUT_DIR / "product_usage_distribution.csv",
    index=False
)

product_usage_churn.to_csv(
    PRODUCT_OUTPUT_DIR / "product_usage_churn.csv",
    index=False
)

product_depth_distribution.to_csv(
    PRODUCT_OUTPUT_DIR / "product_depth_distribution.csv",
    index=False
)

product_depth_churn.to_csv(
    PRODUCT_OUTPUT_DIR / "product_depth_churn.csv",
    index=False
)

product_depth_contribution.to_csv(
    PRODUCT_OUTPUT_DIR / "product_depth_contribution.csv",
    index=False
)

product_utilization_risk_distribution.to_csv(
    PRODUCT_OUTPUT_DIR / "product_utilization_risk_distribution.csv",
    index=False
)

product_utilization_risk_churn.to_csv(
    PRODUCT_OUTPUT_DIR / "product_utilization_risk_churn.csv",
    index=False
)

product_utilization_kpi_summary.to_csv(
    PRODUCT_OUTPUT_DIR / "product_utilization_kpi_summary.csv",
    index=False
)

product_utilization_insight_table.to_csv(
    PRODUCT_OUTPUT_DIR / "product_utilization_insight_table.csv",
    index=False
)

print("Product utilization analysis outputs exported successfully.")
print(f"Output Directory: {PRODUCT_OUTPUT_DIR}")
print("Files Exported: 11")

Product utilization analysis outputs exported successfully.
Output Directory: C:\Users\Mohit\OneDrive\Documents\Data Analytics\Unified Mentor Projects\Customer Engagement & Product Utilization Analytics for Retention Strategy\data\processed\product_utilization_analysis
Files Exported: 11


## Product Utilization Analysis Insight Summary

This section summarizes the key product utilization findings generated in Notebook 05.

The analysis confirms that product depth is strongly connected with customer retention behavior. Customers using more than one product show better retention compared with single-product customers, while customers with very high product load show elevated churn risk.

Key findings:

- Single-product customers represent a large share of the customer base and show weaker retention.
- Multi-product customers show stronger retention and lower churn compared with single-product customers.
- Customers with optimal product depth show the strongest retention profile.
- High product load customers show elevated churn risk, indicating that more products do not always mean better retention.
- Product utilization should be used as a dashboard dimension for customer prioritization, cross-sell planning, and retention monitoring.
- Product depth contribution helps explain how churn is distributed across different product usage groups.

## Dashboard Mapping Table

In [43]:
# Create product utilization dashboard mapping table
product_utilization_dashboard_mapping = pd.DataFrame({
    "Dashboard Module": [
        "Product Utilization Overview",
        "Single vs Multi Product Analysis",
        "Product Depth Retention Analysis",
        "Product Utilization Risk Segments",
        "Product Depth Churn Contribution",
        "Product Utilization KPI Summary"
    ],
    "Notebook Output Used": [
        "product_count_distribution.csv",
        "product_usage_churn.csv",
        "product_depth_churn.csv",
        "product_utilization_risk_distribution.csv",
        "product_depth_contribution.csv",
        "product_utilization_kpi_summary.csv"
    ],
    "Dashboard Purpose": [
        "Show overall customer distribution by number of products",
        "Compare churn and retention between single-product and multi-product customers",
        "Identify which product depth segment has the best or weakest retention profile",
        "Classify customers into product utilization risk groups",
        "Explain how much each product depth segment contributes to total churn",
        "Display executive-level product utilization KPIs"
    ]
})

product_utilization_dashboard_mapping

,Dashboard Module,Notebook Output Used,Dashboard Purpose
0,Product Utilization Overview,product_count_distribution.csv,Show overall customer distribution by number o...
1,Single vs Multi Product Analysis,product_usage_churn.csv,Compare churn and retention between single-pro...
2,Product Depth Retention Analysis,product_depth_churn.csv,Identify which product depth segment has the b...
3,Product Utilization Risk Segments,product_utilization_risk_distribution.csv,Classify customers into product utilization ri...
4,Product Depth Churn Contribution,product_depth_contribution.csv,Explain how much each product depth segment co...
5,Product Utilization KPI Summary,product_utilization_kpi_summary.csv,Display executive-level product utilization KPIs


## Notebook 05 Final Conclusion

Notebook 05 completed the product utilization impact analysis for the Customer Engagement & Product Utilization Analytics project.

The analysis shows that product depth is an important retention signal. Customers with multiple products generally retain better than single-product customers, while customers with optimal product depth show the strongest retention profile.

At the same time, the analysis also confirms that excessive product load can increase churn risk. This means product utilization should not be interpreted only as “more products are always better.” Instead, customer product depth should be evaluated together with engagement, churn status, and relationship strength.

Final analytical outcomes:

- Product count distribution was validated.
- Product depth churn behavior was analyzed.
- Single-product and multi-product customers were compared.
- Optimal product depth retention advantage was calculated.
- Product depth contribution to churn was measured.
- Product utilization risk segments were created.
- KPI summary and insight tables were prepared for dashboard use.
- Product utilization outputs were exported successfully.

The exported outputs are now ready to support the Streamlit dashboard module:

`Product Utilization Impact Overview`

The next notebook will focus on deeper customer segmentation and retention priority analysis.

## Final Checkpoint Confirmation

In [44]:
# Final checkpoint for Notebook 05
print("Notebook 05 completed successfully.")
print("Analysis Area: Product Utilization Impact Analysis")
print(f"Input Dataset Rows: {df.shape[0]:,}")
print(f"Input Dataset Columns: {df.shape[1]:,}")
print("Output Files Exported: 11")
print("Dashboard Module Ready: Product Utilization Impact Overview")

Notebook 05 completed successfully.
Analysis Area: Product Utilization Impact Analysis
Input Dataset Rows: 10,000
Input Dataset Columns: 35
Output Files Exported: 11
Dashboard Module Ready: Product Utilization Impact Overview
